In [1]:
import pandas as pd
import numpy as np
import joblib
import re

from scipy.sparse import csr_matrix, hstack
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [2]:
best_model = joblib.load("best_model.pkl")
tfidf = joblib.load("tfidf_vectorizer.pkl")

In [3]:
def extract_features(text):

    words = text.split()

    char_count = len(text)

    word_count = len(words)

    sentence_count = max(
        len(re.findall(r"[.!?]", text)),
        1
    )

    avg_word_length = np.mean(
        [len(i) for i in words]
    ) if words else 0

    digit_ratio = sum(
        c.isdigit()
        for c in text
    ) / max(len(text),1)

    special_ratio = len(
        re.findall(
            r"[^a-zA-Z0-9\s]",
            text
        )
    ) / max(len(text),1)

    exclamation_count = text.count("!")

    question_count = text.count("?")

    has_email = int(
        bool(
            re.search(
                r"\S+@\S+",
                text
            )
        )
    )

    gmail_email = int(
        "@gmail" in text.lower()
    )

    yahoo_email = int(
        "@yahoo" in text.lower()
    )

    outlook_email = int(
        "@outlook" in text.lower()
    )

    salary_present = int(
        bool(
            re.search(
                r"\d",
                text
            )
        )
    )

    salary_daily = int(
        "day" in text.lower()
    )

    salary_monthly = int(
        "month" in text.lower()
    )

    salary_hourly = int(
        "hour" in text.lower()
    )

    avg_sentence_length = min(
        word_count/sentence_count,
        100
    )

    unique_word_ratio = len(
        set(words)
    ) / max(word_count,1)

    repetition_score = (
        1-unique_word_ratio
    )

    long_word_ratio = sum(
        len(i)>8
        for i in words
    ) / max(word_count,1)

    numeric_token_ratio = sum(
        any(c.isdigit() for c in i)
        for i in words
    ) / max(word_count,1)

    return [

        char_count,

        word_count,

        sentence_count,

        avg_word_length,

        digit_ratio,

        special_ratio,

        exclamation_count,

        question_count,

        has_email,

        gmail_email,

        yahoo_email,

        outlook_email,

        salary_present,

        salary_daily,

        salary_monthly,

        salary_hourly,

        avg_sentence_length,

        unique_word_ratio,

        repetition_score,

        long_word_ratio,

        numeric_token_ratio

    ]

In [6]:
import pandas as pd

external_df = pd.read_excel("external_test.xlsx")
external_df.head()


,job_title,company_name,job_description,label
0,Backend Engineer,NovaTech Systems,Build scalable REST APIs using Python and Post...,0
1,Data Analyst,Insight Metrics,Analyze dashboards SQL queries and BI reports....,0
2,ML Intern,VisionAI Labs,Assist with model evaluation preprocessing pip...,0
3,Frontend Developer,PixelCraft,Develop React interfaces with responsive desig...,0
4,DevOps Engineer,CloudStack,Maintain CI CD pipelines and Kubernetes infras...,0


In [7]:
external_df["combined_text"] = (

    external_df["job_title"].fillna("")

    + " "

    + external_df["company_name"].fillna("")

    + " "

    + external_df["job_description"].fillna("")

)

In [8]:
X_text = tfidf.transform(

    external_df["combined_text"]

)

In [9]:
manual = external_df["combined_text"].apply(

    extract_features

)

manual = pd.DataFrame(

    manual.tolist(),

    columns=[

        "char_count",
        "word_count",
        "sentence_count",
        "avg_word_length",
        "digit_ratio",
        "special_ratio",
        "exclamation_count",
        "question_count",

        "has_email",
        "gmail_email",
        "yahoo_email",
        "outlook_email",

        "salary_present",
        "salary_daily",
        "salary_monthly",
        "salary_hourly",

        "avg_sentence_length",
        "unique_word_ratio",
        "repetition_score",
        "long_word_ratio",
        "numeric_token_ratio"

    ]

)

X_manual = csr_matrix(

    manual.values

)

In [10]:
X = hstack([

    X_text,

    X_manual

])

In [11]:
X = hstack([

    X_text,

    X_manual

])

In [12]:
pred = best_model.predict(X)

prob = best_model.predict_proba(X)[:,1]

external_df["Prediction"] = pred

external_df["Probability"] = prob

In [13]:
print(

classification_report(

external_df["label"],

pred

)

)

print()

print(

confusion_matrix(

external_df["label"],

pred

)

)

print()

print(

"Accuracy :",

accuracy_score(

external_df["label"],

pred

)

)

print(

"Precision :",

precision_score(

external_df["label"],

pred

)

)

print(

"Recall :",

recall_score(

external_df["label"],

pred

)

)

print(

"F1 :",

f1_score(

external_df["label"],

pred

)

)

              precision    recall  f1-score   support

           0       0.50      1.00      0.67        10
           1       0.00      0.00      0.00        10

    accuracy                           0.50        20
   macro avg       0.25      0.50      0.33        20
weighted avg       0.25      0.50      0.33        20


[[10  0]
 [10  0]]

Accuracy : 0.5
Precision : 0.0
Recall : 0.0
F1 : 0.0


C:\Users\Raghvendra Goyal\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Raghvendra Goyal\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Raghvendra Goyal\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\si

In [14]:
wrong = external_df[

external_df["label"]

!=

external_df["Prediction"]

]

print(

wrong.shape

)

wrong.head(20)

(10, 7)


,job_title,company_name,job_description,label,combined_text,Prediction,Probability
10,Work From Home Data Entry,QuickCash Careers,Earn 5000 daily No interview WhatsApp HR now L...,1,Work From Home Data Entry QuickCash Careers Ea...,0,0.173313
11,Typing Job,Easy Money Hub,Registration fee 499 required before joining I...,1,Typing Job Easy Money Hub Registration fee 499...,0,0.358900
12,Part Time Freelancer,Online Income Pvt,Earn 3000 per hour with zero skills Telegram r...,1,Part Time Freelancer Online Income Pvt Earn 30...,0,0.038194
13,Captcha Entry,Digital Fast Jobs,Unlimited income from captcha typing Security ...,1,Captcha Entry Digital Fast Jobs Unlimited inco...,0,0.006782
14,Airport Staff Urgent,Sky Hiring,Direct joining today Processing fee mandatory ...,1,Airport Staff Urgent Sky Hiring Direct joining...,0,0.052347
15,Amazon Review Job,Review Club,Like products and earn daily Joining fee only 299,1,Amazon Review Job Review Club Like products an...,0,0.092304
16,Crypto Promotion Executive,Coin Boost,Receive salary in crypto after account activat...,1,Crypto Promotion Executive Coin Boost Receive ...,0,0.004645
17,WFH Executive,Remote Biz,No experience needed Earn 8000 day Pay verific...,1,WFH Executive Remote Biz No experience needed ...,0,0.110901
18,Virtual Assistant,Home Desk,Guaranteed income Limited slots Message on Tel...,1,Virtual Assistant Home Desk Guaranteed income ...,0,0.007594
19,HR Recruiter,Instant Hire,100 percent selection Deposit refundable onboa...,1,HR Recruiter Instant Hire 100 percent selectio...,0,0.085475


In [15]:
print(
classification_report(
external_df["label"],
pred
)
)

              precision    recall  f1-score   support

           0       0.50      1.00      0.67        10
           1       0.00      0.00      0.00        10

    accuracy                           0.50        20
   macro avg       0.25      0.50      0.33        20
weighted avg       0.25      0.50      0.33        20



C:\Users\Raghvendra Goyal\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Raghvendra Goyal\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Raghvendra Goyal\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\si

In [16]:
print(wrong.shape)

wrong[
[
"job_title",
"company_name",
"label",
"Prediction",
"Probability"
]
]

(10, 7)


,job_title,company_name,label,Prediction,Probability
10,Work From Home Data Entry,QuickCash Careers,1,0,0.173313
11,Typing Job,Easy Money Hub,1,0,0.358900
12,Part Time Freelancer,Online Income Pvt,1,0,0.038194
13,Captcha Entry,Digital Fast Jobs,1,0,0.006782
14,Airport Staff Urgent,Sky Hiring,1,0,0.052347
15,Amazon Review Job,Review Club,1,0,0.092304
16,Crypto Promotion Executive,Coin Boost,1,0,0.004645
17,WFH Executive,Remote Biz,1,0,0.110901
18,Virtual Assistant,Home Desk,1,0,0.007594
19,HR Recruiter,Instant Hire,1,0,0.085475


In [17]:
print(
confusion_matrix(
external_df["label"],
pred
)
)

[[10  0]
 [10  0]]


In [18]:
external_df[
[
"job_title",
"Prediction",
"Probability"
]
].head(20)

,job_title,Prediction,Probability
0,Backend Engineer,0,0.185911
1,Data Analyst,0,0.005523
2,ML Intern,0,0.062567
3,Frontend Developer,0,0.009424
4,DevOps Engineer,0,0.009671
5,Software Engineer,0,0.071294
6,Cybersecurity Analyst,0,0.008962
7,QA Engineer,0,0.016208
8,AI Research Intern,0,0.009671
9,Java Developer,0,0.010526


In [19]:
print(
external_df["Probability"].describe()
)

count    20.000000
mean      0.066011
std       0.088398
min       0.004645
25%       0.009309
50%       0.027201
75%       0.087182
max       0.358900
Name: Probability, dtype: float64


In [20]:
fake_pred = external_df[
external_df["Prediction"]==1
]

real_pred = external_df[
external_df["Prediction"]==0
]

print(fake_pred.shape)
print(real_pred.shape)

(0, 7)
(20, 7)
